# Shallow Diffuse 主表 baseline 单独真实 GPU 链路入口

该 Notebook 用于 Colab 冷启动, 只运行 `shallow_diffuse` 一个主表 external baseline, 避免四个 baseline 在同一会话中串行运行导致超过 Colab 最大运行时间。正式逻辑位于 `paper_workflow/colab_utils/external_baseline_method_faithful.py`, Notebook 只作为远程执行入口。


## 运行前准备

1. 在 Colab 中选择 GPU runtime。
2. 确认 Hugging Face 账号已获得 `stabilityai/stable-diffusion-3.5-medium` 访问权限, 并配置 `HF_TOKEN`。
3. 默认产物写入 `/content/drive/MyDrive/SLM/<运行层级>_results/external_baseline_method_faithful`。
4. 每个单 baseline 入口都会写出 `split_observations/<baseline_id>_baseline_observations.json`, 后续结果闭合会自动合并这些拆分产物。


In [ ]:
import os

from google.colab import drive

drive.mount('/content/drive')
# 修改为 "full_paper" 可切换完整论文运行层级。
SLM_WM_PAPER_RUN_NAME = "pilot_paper"
os.environ["SLM_WM_PAPER_RUN_NAME"] = SLM_WM_PAPER_RUN_NAME
SLM_WM_PRIMARY_BASELINE_METHODS = "shallow_diffuse"
os.environ["SLM_WM_PRIMARY_BASELINE_METHODS"] = SLM_WM_PRIMARY_BASELINE_METHODS


In [ ]:
%pip install -q --upgrade diffusers transformers accelerate safetensors sentencepiece protobuf huggingface_hub open_clip_torch scikit-learn scipy pandas datasets tqdm


In [ ]:
import os
import subprocess
import sys

repository_url = os.environ.get('SLM_WM_REPOSITORY_URL', 'https://github.com/RICHAAARC/SLM-WM.git')
repository_ref = os.environ.get('SLM_WM_REPOSITORY_REF', 'main')
workspace_dir = os.environ.get('SLM_WM_WORKSPACE_DIR', '/content/slm_wm_repository')

if not os.path.exists(workspace_dir):
    subprocess.run(['git', 'clone', repository_url, workspace_dir], check=True)

subprocess.run(['git', 'fetch', '--all'], cwd=workspace_dir, check=True)
subprocess.run(['git', 'checkout', repository_ref], cwd=workspace_dir, check=True)
os.chdir(workspace_dir)
if workspace_dir not in sys.path:
    sys.path.insert(0, workspace_dir)
print({'workspace_dir': workspace_dir, 'repository_ref': repository_ref})

from paper_workflow.colab_utils.paper_run_environment import configure_paper_run_environment
paper_run_environment = configure_paper_run_environment(
    workflow_name="external_baseline_method_faithful",
    baseline_id=os.environ["SLM_WM_PRIMARY_BASELINE_METHODS"],
)
print(paper_run_environment)


In [ ]:
import os

try:
    from google.colab import userdata
    token_from_secret = userdata.get('HF_TOKEN')
except Exception:
    token_from_secret = None

if token_from_secret and not os.environ.get('HF_TOKEN'):
    os.environ['HF_TOKEN'] = token_from_secret

if os.environ.get('HF_TOKEN'):
    from huggingface_hub import login
    login(token=os.environ['HF_TOKEN'])
    print('huggingface_login_ready')
else:
    print('HF_TOKEN_not_found')


In [ ]:
import torch

device_report = {
    'cuda_available': torch.cuda.is_available(),
    'device_count': torch.cuda.device_count(),
    'device_name': torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none',
}
print(device_report)
assert device_report['cuda_available'], '需要 Colab GPU runtime'


In [ ]:
from paper_workflow.colab_utils.external_baseline_method_faithful import run_default_external_baseline_method_faithful_plan

external_baseline_method_faithful_summary = run_default_external_baseline_method_faithful_plan(root='.')
external_baseline_method_faithful_summary


In [ ]:
from pathlib import Path

summary_path = Path('outputs/external_baseline_method_faithful/external_baseline_method_faithful_summary.json')
prior_manifest_path = Path('outputs/external_baseline_method_faithful/external_baseline_method_faithful_prior_package_manifest.json')
source_prepare_path = Path('outputs/external_baseline_method_faithful/t2smark_source_prepare_result.json')
print(summary_path.read_text(encoding='utf-8') if summary_path.exists() else external_baseline_method_faithful_summary)
for diagnostic_path in (prior_manifest_path, source_prepare_path):
    if diagnostic_path.exists():
        print(diagnostic_path)
        print(diagnostic_path.read_text(encoding='utf-8')[:4000])


In [ ]:
import os
import subprocess
from datetime import datetime, timezone

from paper_workflow.colab_utils.external_baseline_method_faithful import package_external_baseline_method_faithful_outputs


def resolve_short_commit():
    try:
        result = subprocess.run(['git', 'rev-parse', '--short', 'HEAD'], check=True, capture_output=True, text=True)
    except Exception:
        return 'git_unknown'
    return result.stdout.strip() or 'git_unknown'


drive_output_dir = os.environ['SLM_WM_EXTERNAL_BASELINE_DRIVE_OUTPUT_DIR']
selected_baseline_id = os.environ['SLM_WM_PRIMARY_BASELINE_METHODS'].replace(',', '_')
utc_suffix = datetime.now(timezone.utc).strftime('%Y%m%dt%H%M%sz')
short_commit = resolve_short_commit()
archive_name = f'external_baseline_method_faithful_package_{selected_baseline_id}_{utc_suffix}_{short_commit}.zip'
archive_record = package_external_baseline_method_faithful_outputs(root='.', drive_output_dir=drive_output_dir, archive_name=archive_name)
archive_record.to_dict()


In [ ]:
from pathlib import Path
import json
import os

output_root = Path('outputs/external_baseline_method_faithful')
selected_baseline_ids = {item.strip() for item in os.environ['SLM_WM_PRIMARY_BASELINE_METHODS'].split(',') if item.strip()}
expected_sample_count = paper_run_expected_sample_count
summary_metadata = external_baseline_method_faithful_summary.get('metadata', {})
observation_path_candidates = [
    external_baseline_method_faithful_summary.get('baseline_observations_path'),
    summary_metadata.get('baseline_observations_path') if isinstance(summary_metadata, dict) else '',
    output_root / 'execution/baseline_observations.json',
]
observations_path = None
for observation_path_candidate in observation_path_candidates:
    if not observation_path_candidate:
        continue
    candidate_path = Path(observation_path_candidate)
    if candidate_path.is_file():
        observations_path = candidate_path
        break
assert observations_path is not None, {
    'missing_baseline_observations_candidates': [str(candidate) for candidate in observation_path_candidates if candidate],
}
observation_rows = json.loads(observations_path.read_text(encoding='utf-8'))
print(f'读取 baseline observations: {observations_path}')
print(observations_path.read_text(encoding='utf-8')[:4000])

assert external_baseline_method_faithful_summary['run_decision'] == 'pass', external_baseline_method_faithful_summary
assert external_baseline_method_faithful_summary['external_baseline_method_faithful_ready'] is True, external_baseline_method_faithful_summary
assert external_baseline_method_faithful_summary['adapter_execution_ready'] is True, external_baseline_method_faithful_summary
assert external_baseline_method_faithful_summary['adapter_observation_count'] > 0, external_baseline_method_faithful_summary
assert external_baseline_method_faithful_summary['primary_baseline_adapter_ready'] is True, external_baseline_method_faithful_summary
assert set(external_baseline_method_faithful_summary['primary_baseline_ids']) == selected_baseline_ids, external_baseline_method_faithful_summary
assert external_baseline_method_faithful_summary['expected_sample_count'] == expected_sample_count, external_baseline_method_faithful_summary
minimum_observation_count = expected_sample_count * len(selected_baseline_ids) * 2
assert external_baseline_method_faithful_summary['primary_baseline_observation_count'] >= minimum_observation_count, external_baseline_method_faithful_summary
for baseline_id in selected_baseline_ids:
    split_path = output_root / 'split_observations' / f'{baseline_id}_baseline_observations.json'
    assert split_path.is_file(), f'单 baseline observation 拆分文件缺失: {split_path}'
    split_rows = json.loads(split_path.read_text(encoding='utf-8'))
    assert len(split_rows) >= expected_sample_count * 2, {'baseline_id': baseline_id, 'split_row_count': len(split_rows)}

if 't2smark' in selected_baseline_ids:
    assert external_baseline_method_faithful_summary['t2smark_selected'] is True, external_baseline_method_faithful_summary
    assert external_baseline_method_faithful_summary['t2smark_real_method_faithful_ready'] is True, external_baseline_method_faithful_summary
    assert external_baseline_method_faithful_summary['t2smark_result_count'] >= expected_sample_count, external_baseline_method_faithful_summary
    assert external_baseline_method_faithful_summary['image_pair_count'] >= expected_sample_count, external_baseline_method_faithful_summary
else:
    assert external_baseline_method_faithful_summary['t2smark_selected'] is False, external_baseline_method_faithful_summary
    attack_env_by_baseline = {
        'tree_ring': 'SLM_WM_TREE_RING_ATTACK_FAMILIES',
        'gaussian_shading': 'SLM_WM_GAUSSIAN_SHADING_ATTACK_FAMILIES',
        'shallow_diffuse': 'SLM_WM_SHALLOW_DIFFUSE_ATTACK_FAMILIES',
    }
    for baseline_id in selected_baseline_ids:
        manifest_path = output_root / 'adapter_outputs' / baseline_id / f'{baseline_id}_method_faithful_sd35_adapter_manifest.json'
        image_pairs_path = output_root / 'adapter_outputs' / baseline_id / 'artifacts' / f'{baseline_id}_image_pairs.json'
        attacked_manifest_path = output_root / 'adapter_outputs' / baseline_id / 'artifacts' / 'attacked_image_manifest.json'
        assert manifest_path.is_file(), f'{baseline_id} method-faithful manifest 缺失: {manifest_path}'
        assert image_pairs_path.is_file(), f'{baseline_id} image pairs 缺失: {image_pairs_path}'
        assert attacked_manifest_path.is_file(), f'{baseline_id} attacked manifest 缺失: {attacked_manifest_path}'
        attacked_manifest = json.loads(attacked_manifest_path.read_text(encoding='utf-8'))
        attack_count = len([item for item in os.environ[attack_env_by_baseline[baseline_id]].split(',') if item])
        expected_attacked_count = attack_count * expected_sample_count * 2
        assert attacked_manifest.get('attacked_image_count', 0) >= expected_attacked_count, attacked_manifest
